# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the
web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON
schema)
2. A function or coroutine to execute.

In [51]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")    # type: ignore

model = init_chat_model("groq:openai/gpt-oss-120b")

In [ ]:
from langchain.tools import tool

@tool
def get_skills(name: str) -> str | None:
    """Get skills of a player by name."""
    player_skill = {
        "samin": ["dribbling", "finishing", "passing", "vision", "ball control", "free kick", "acceleration"],
        "messi": ["passing", "vision", "ball control", "dribbling", "playmaking", "through balls", "positional awareness"],
        "iniesta": ["dribbling", "passing", "vision", "ball control", "playmaking", "through balls", "close control"],
        "pique": ["defending", "tackling", "heading", "strength", "positioning", "passing", "aerial duels"],
        "villa": ["finishing", "shooting", "positioning", "dribbling", "pace", "off the ball movement", "volleys"]
    }

    skill_list =  player_skill.get(name.lower(), None)    # GROQ can't acces list, it can access string only
    return ", ".join(skill_list)    # type: ignore

model_with_tools = model.bind_tools([get_skills])

In [64]:
response = model_with_tools.invoke("What are the skills of samin?")
print(response)


for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get skills of "samin". Use function get_skills.', 'tool_calls': [{'id': 'fc_65733058-cef8-4013-bc3e-faad76a4843f', 'function': {'arguments': '{"name":"samin"}', 'name': 'get_skills'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 130, 'total_tokens': 176, 'completion_time': 0.096618051, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.005214251, 'prompt_tokens_details': None, 'queue_time': 0.335342988, 'total_time': 0.101832302}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_77b12279f9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09b3a-ea18-7b31-a3a3-916d48b17d15-0' tool_calls=[{'name': 'get_skills', 'args': {'name': 'samin'}, 'id': 'fc_65733058-cef8-4013-bc3e-faad76a4843f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 130, 'ou

### Tool Execution Loops

In [66]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What are the skills of Samin?"}]
ai_messages = model_with_tools.invoke(messages)
messages.append(ai_messages)    # type: ignore

# Step 2: Execute tools and collect results
for tool_call in ai_messages.tool_calls:
    tool_result = get_skills.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass Result back to the result for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

Samin’s skill set includes:

- Dribbling  
- Finishing  
- Passing  
- Vision  
- Ball control  
- Free‑kick taking  
- Acceleration


In [68]:
print(messages)

[{'role': 'user', 'content': 'What are the skills of Samin?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What are the skills of Samin?" Likely need to call get_skills function with name "Samin".', 'tool_calls': [{'id': 'fc_20ef72f7-204a-4b67-bc00-f5b04c050d2a', 'function': {'arguments': '{"name":"Samin"}', 'name': 'get_skills'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 130, 'total_tokens': 187, 'completion_time': 0.11925603, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.041867918, 'prompt_tokens_details': None, 'queue_time': 0.500543861, 'total_time': 0.161123948}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09b3c-2384-7771-ae9a-0e2a7506b161-0', tool_calls=[{'name': 'get_skills', 'args': {'name': 'Samin'}, 'id': 'fc_20ef